# Hunyuan3D Avatar 3D (Colab)

Pipeline principal pour Colab T4: Hunyuan3D-2mini pour la forme, puis Hunyuan3D-Paint pour la texture.

In [ ]:
import shutil
import subprocess

if shutil.which('nvidia-smi') is None:
    print('nvidia-smi introuvable: runtime probablement en mode CPU. Passe en GPU T4 puis redemarre le runtime.')
else:
    subprocess.run(['nvidia-smi'], check=False)


In [ ]:
from pathlib import Path
from google.colab import files

PNG_CANDIDATES = [
    Path('/content/avatar-man-1.png'),
]
PNG_CANDIDATES += sorted(Path('/content').glob('*.png'))

existing = next((p for p in PNG_CANDIDATES if p.exists()), None)
if existing is None:
    print('Aucun PNG detecte dans /content. Upload du fichier image...')
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('Aucun fichier upload.')
    first_name = next(iter(uploaded.keys()))
    existing = Path('/content') / first_name

print(f'Image selectionnee: {existing}')
SELECTED_IMAGE = str(existing)


In [ ]:
INPUT_IMAGE = SELECTED_IMAGE
OUT_DIR = '/content/outputs/avatar-man-1-hunyuan'
SEED = 12345
OCTREE_RESOLUTION = 320

# Memory presets for T4
# MAX_QUALITY: NUM_INFERENCE_STEPS=35, NUM_CHUNKS=20000
# BALANCED:    NUM_INFERENCE_STEPS=28, NUM_CHUNKS=12000
# SAFE:        NUM_INFERENCE_STEPS=24, NUM_CHUNKS=8000

NUM_INFERENCE_STEPS = 28
NUM_CHUNKS = 12000


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

REPO_DIR = Path('/content/Hunyuan3D-2')
ZIP_CANDIDATES = [
    Path('/content/Hunyuan3D-2.zip'),
    Path('/content/Hunyuan3D-2-main.zip'),
]

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

def run(cmd: str, check: bool = True) -> bool:
    print(f'\n$ {cmd}')
    result = subprocess.run(cmd, shell=True)
    if check and result.returncode != 0:
        raise RuntimeError(f'Command failed: {cmd}')
    return result.returncode == 0

# 1) Primary path: GitHub clone
cloned = run('git clone --depth 1 https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git /content/Hunyuan3D-2', check=False)

# 2) Fallback path: local zip upload (no GitHub required)
if not cloned:
    print('GitHub inaccessible from this Colab runtime. Trying local zip fallback...')
    zip_path = next((z for z in ZIP_CANDIDATES if z.exists()), None)
    if zip_path is None:
        raise RuntimeError(
            'GitHub unreachable and no local zip found. Upload Hunyuan3D-2-main.zip to /content, then rerun this cell.'
        )
    run(f'unzip -q {zip_path} -d /content')
    extracted = Path('/content/Hunyuan3D-2-main')
    if not extracted.exists():
        raise RuntimeError('Zip extracted but /content/Hunyuan3D-2-main was not found.')
    extracted.rename(REPO_DIR)

assert REPO_DIR.exists(), 'Repository setup failed.'

# Install dependencies (pinned for torch 2.4.x compatibility on Colab T4)
run('cd /content/Hunyuan3D-2 && python3 -m pip install --upgrade pip')
run('cd /content/Hunyuan3D-2 && python3 -m pip install torch==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu121')
run('cd /content/Hunyuan3D-2 && python3 -m pip install -r requirements.txt')
run('cd /content/Hunyuan3D-2 && python3 -m pip install diffusers==0.29.2')
run('cd /content/Hunyuan3D-2 && python3 -m pip install -e .')

# Build custom extensions
run('cd /content/Hunyuan3D-2/hy3dgen/texgen/custom_rasterizer && python3 setup.py install')
run('cd /content/Hunyuan3D-2/hy3dgen/texgen/differentiable_renderer && python3 setup.py install')

# Quick sanity check
run('python3 - <<'PY'\nimport torch, diffusers\nprint('torch', torch.__version__)\nprint('diffusers', diffusers.__version__)\nPY')

print('Environment ready.')


In [ ]:
import gc
import json
import os
import shutil
import time
from pathlib import Path

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import torch
from PIL import Image

os.chdir('/content/Hunyuan3D-2')

from hy3dgen.rembg import BackgroundRemover
from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline

out_dir = Path(OUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

raw = Image.open(INPUT_IMAGE)
image = raw.convert('RGBA')
if raw.mode == 'RGB':
    image = BackgroundRemover()(image)

started = time.time()
shape_pipeline = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
    'tencent/Hunyuan3D-2mini',
    subfolder='hunyuan3d-dit-v2-mini',
    variant='fp16',
)
shape_pipeline.enable_model_cpu_offload()

mesh = shape_pipeline(
    image=image,
    num_inference_steps=NUM_INFERENCE_STEPS,
    octree_resolution=OCTREE_RESOLUTION,
    num_chunks=NUM_CHUNKS,
    generator=torch.manual_seed(SEED),
    output_type='trimesh',
)[0]

shape_glb = out_dir / 'avatar-man-1_hunyuan_shape.glb'
mesh.export(shape_glb)

del shape_pipeline
torch.cuda.empty_cache()
gc.collect()

texture_status = 'SKIPPED'
final_glb = shape_glb
texture_error = None
try:
    from hy3dgen.texgen import Hunyuan3DPaintPipeline

    paint_pipeline = Hunyuan3DPaintPipeline.from_pretrained('tencent/Hunyuan3D-2')
    paint_pipeline.enable_model_cpu_offload()
    textured_mesh = paint_pipeline(mesh, image=image)
    final_glb = out_dir / 'avatar-man-1_hunyuan_textured.glb'
    textured_mesh.export(final_glb)
    texture_status = 'PASS'
except Exception as exc:
    texture_status = 'FAIL'
    texture_error = f'{type(exc).__name__}: {exc}'

manifest = {
    'pipeline': 'Hunyuan3D-2mini + Hunyuan3D-Paint',
    'input_image': INPUT_IMAGE,
    'seed': SEED,
    'octree_resolution': OCTREE_RESOLUTION,
    'num_inference_steps': NUM_INFERENCE_STEPS,
    'num_chunks': NUM_CHUNKS,
    'shape_glb': str(shape_glb),
    'final_glb': str(final_glb),
    'texture_status': texture_status,
    'texture_error': texture_error,
    'elapsed_seconds': round(time.time() - started, 2),
}
(out_dir / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')

delivery_dir = out_dir / 'avatar-man-1_hunyuan_delivery'
if delivery_dir.exists():
    shutil.rmtree(delivery_dir)
delivery_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(out_dir / 'manifest.json', delivery_dir / 'manifest.json')
shutil.copy2(final_glb, delivery_dir / final_glb.name)
if final_glb != shape_glb:
    shutil.copy2(shape_glb, delivery_dir / shape_glb.name)

zip_path = shutil.make_archive(str(out_dir / 'avatar-man-1_hunyuan_delivery'), 'zip', root_dir=str(delivery_dir))
print(json.dumps({'manifest': manifest, 'zip': zip_path}, indent=2))


In [ ]:
!ls -lah /content/outputs/avatar-man-1-hunyuan
from google.colab import files
files.download('/content/outputs/avatar-man-1-hunyuan/avatar-man-1_hunyuan_delivery.zip')